# Project FORESIGHT — Baseline Forecast

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "../data"
weekly = pd.read_pickle(os.path.join(DATA_DIR,"weekly_sales.pkl"))
weekly["week_start"] = pd.to_datetime(weekly["week_start"])
weekly = weekly.sort_values(["sku_id","week_start"])

In [ ]:
def wape(actual, forecast):
    denominator = np.sum(np.abs(actual))
    return np.nan if denominator == 0 else np.sum(np.abs(actual-forecast))/denominator

In [ ]:
# Seasonal-naive: use 52 weeks when enough history exists; otherwise previous week
max_history = weekly.groupby("sku_id")["week_start"].nunique().max()
lag = 52 if max_history >= 104 else 1
weekly["baseline_forecast"] = weekly.groupby("sku_id")["units_sold"].shift(lag)
print("Seasonal lag:", lag, "weeks")

In [ ]:
# Last 20% chronological holdout
weeks = np.sort(weekly["week_start"].unique())
cutoff = weeks[max(0,int(len(weeks)*0.8)-1)]
test = weekly[(weekly["week_start"] > cutoff) & weekly["baseline_forecast"].notna()].copy()

score = wape(test["units_sold"], test["baseline_forecast"])
print(f"Baseline WAPE: {score:.4f}")
print(f"Baseline WAPE %: {score*100:.2f}%")

In [ ]:
weekly.to_pickle(os.path.join(DATA_DIR,"weekly_baseline.pkl"))
test.to_pickle(os.path.join(DATA_DIR,"baseline_test.pkl"))
print("Baseline results saved.")